# Notebook 1: Procesado del archivo twitter

En esta primer notebook, se explica la estructura de la red que hemos usado para este trabajo, que sacamos del datset de la universidad de Stanford. También se explican las decisiones tomadas a la hora de limpiar los datos para poder explotarlos.

## Estructura del Grafo de Twitter de SNAP (Stanford)

### Archivos 
El id de cada 'conjunto' de archivos
- **`.edges`**: Lista de conexiones dirigidas entre usuarios (ej: `usuario_origen usuario_destino`). Representan menciones, follows o retweets.
- **`.feat`**: Matriz binaria donde cada fila corresponde a un usuario y cada columna a una característica (hashtags, intereses).
- **`.featnames`**: Describe el significado de cada característica en `.feat` (ej: `0 #politics`).
- **`.egofeat`**: Características del nodo central ("ego") del dataset.

### Componentes clave
1. **Nodos**: Usuarios únicos identificados por ID.
2. **Aristas**: Conexiones dirigidas (importa la dirección).
3. **Atributos**:
   - Asignados desde `.feat` (ej: hashtags usados).
   - El "ego" (usuario central) tiene características especiales en `.egofeat`.

### Cómo se interpreta
- **Ejemplo de `.edges`**: 123 456 -> El usuario 123 interactuó con 456
- **Ejemplo de `.feat`**: 1 0 1 -> Usuario usa hashtags 0 y 2
- **Ejemplo de `.featnames`**:  0 #hashtag_politics, 1 #hashtag_tech, 2 #hashtag_sports

## Obtención de user_hashtags a partir de los datos de la carpeta twitter

Por motivos de capacidad computacional (tanto del ordenador como de la LLM que usaremos posteriormente), seleccionaremos únicamente **10 ids distintos** para el análisis.

### Proceso de creación del archivo:
1. Extraeremos los hashtags utilizados por los usuarios de las comunidades asociadas a cada id.
2. Utilizaremos los archivos `.feat` y `.featnames` para identificar estos hashtags.
3. Generaremos un archivo `user_hashtags.csv` con esta información estructurada.

### Justificación del enfoque alternativo:
El grafo **no se construirá a partir de las conexiones del archivo `.edges`** porque:
- Con solo 10 ids, obtendríamos **comunidades completamente aisladas**.
- El resultado sería un grafo sin conexiones útiles para el análisis.

### Solución implementada:
- **Estructura del grafo alternativo**:
  - *Nodos*: Usuarios individuales
  - *Aristas*: Conexiones basadas en **intereses compartidos** (hashtags en común)
  
Este enfoque se implementa en el notebook:  
**`2-analisis_redsocial_twitter1.ipynb`**

In [2]:
import numpy as np
import pandas as pd
import os
import json
import matplotlib.pyplot as plt

In [5]:
files_location = "twitter/"
users = pd.Series(os.listdir(files_location)).str.extract(r'(\d+)\.').drop_duplicates().reset_index(drop=True)[0]

In [ ]:
# ------------------------------------
# 1. Load Data
# ------------------------------------
def get_user_interactions(user):
    ego_user_id = "4o Cuatrimestre/Computación Social y Personalización/Trabajo/twitter/" + user
    featnames_path = f"{ego_user_id}.featnames"
    ego_feat_path = f"{ego_user_id}.egofeat"
    circle_feat_paths = f"{ego_user_id}.feat" 
    all_terms = pd.read_csv(featnames_path, sep=" ", usecols=[1], encoding="utf-8", header=None).iloc[:, 0]
    
    #Get boolean matrix for each user in ego's circle.
    ego = pd.concat((pd.Series([user]), pd.read_csv(ego_feat_path, sep=" ", encoding="utf-8", header=None)), axis=1)
    ego.columns = range(ego.shape[1])
    circle = pd.read_csv(circle_feat_paths, sep=" ", encoding="utf-8", header=None)
    features = pd.concat((ego, circle), axis=0).reset_index(drop=True)
    
    # Convert boolean vector for each user into the hashtags they use.
    features_names = pd.concat((features[0], features.apply(lambda x: tuple(all_terms[(x[1:] == 1).reset_index(drop=True)]), axis=1)), axis=1)
    features_names.columns = ['user', 'hashtags/mentions']
    return features_names

In [ ]:
n_users = 10 # 973

In [ ]:
user_hashtags = pd.concat(users[:n_users].apply(get_user_interactions).to_list()).reset_index(drop=True)

In [ ]:
user_hashtags = user_hashtags.groupby('user')['hashtags/mentions'].apply(lambda tuplas: tuple(set(h for t in tuplas for h in t))).reset_index()

In [ ]:
user_hashtags.to_csv("4o Cuatrimestre/Computación Social y Personalización/Trabajo/user_hashtags.csv", index=False, encoding="utf-8")